In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # single-GPU only: prevents Trainer's automatic
                                            # DataParallel wrapping across T4 x2, which breaks
                                            # with 4-bit quantized weights
print("CUDA_VISIBLE_DEVICES set to:", os.environ["CUDA_VISIBLE_DEVICES"])


## 1. Install dependencies

In [ ]:
import subprocess

_installed_torch = subprocess.run(
    ["python", "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True
).stdout.strip()
print("Kaggle-preinstalled torch:", _installed_torch)


In [ ]:
# Note: torch is pinned to the version already in the image so it is
# never silently upgraded/downgraded by this install step.
!pip install -q "transformers>=4.46.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" \
    "accelerate>=1.0.0" "datasets>=3.0.0" "trl>=0.11.0" evaluate scikit-learn sentencepiece \
    "torch=={_installed_torch}"


In [ ]:
import torch

print("torch version:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available to torch. Check that the notebook's Accelerator is set to "
        "GPU (Settings > Accelerator), then Restart Session and re-run all cells."
    )


try:
    torch.cuda.reset_peak_memory_stats()
    print("torch.cuda.reset_peak_memory_stats() OK")
except AttributeError as e:
    raise RuntimeError(
        "torch installed without CUDA memory-stat support. This usually means pip "
        "upgraded torch during the install step. Restart the kernel and re-run this "
        "notebook top-to-bottom without adding -U/--upgrade to the pip install line."
    ) from e


## 2. Imports, seed, and sweep configuration

In [ ]:
import os, gc, json, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- Config ----------------
MODEL_ID     = "Qwen/Qwen2.5-3B"
DATASET_ID   = "FinGPT/fingpt-finred-re"      # FinRED relation-extraction split, HF mirror
OUTPUT_ROOT  = "/kaggle/working/qlora_finred"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

LORA_RANKS      = 16                 
LORA_ALPHA_MULT = 2                    # alpha = r * LORA_ALPHA_MULT (common QLoRA heuristic)
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

MAX_LEN     = 512
NUM_EPOCHS  = 2
BATCH_SIZE  = 4
GRAD_ACCUM  = 4
LR          = 2e-4
MAX_TRAIN_SAMPLES = None   
MAX_EVAL_SAMPLES  = 500    # cap test-set size for generation-based F1 eval (speed)

USE_GRADIENT_CHECKPOINTING = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs:", torch.cuda.device_count() if DEVICE == "cuda" else 0)

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    USE_BF16 = cc_major >= 8
else:
    gpu_name = "cpu"
    cc_major, cc_minor = (0, 0)
    USE_BF16 = False
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {gpu_name} | compute capability: {cc_major}.{cc_minor} | "
      f"hardware bf16: {USE_BF16} -> using compute dtype {COMPUTE_DTYPE}")

# Pin the model to a single GPU rather than "auto". 
DEVICE_MAP = {"": 0} if DEVICE == "cuda" else "cpu"


## 3. Load FinRED and inspect schema

`FinGPT/fingpt-finred-re` follows the Alpaca-style instruction schema (`instruction` / `input` / `output`). We load it defensively and fall back to whatever text columns are actually present, since dataset revisions vary.

In [ ]:
raw = load_dataset(DATASET_ID)
print(raw)

split_name = "train" if "train" in raw else list(raw.keys())[0]
sample = raw[split_name][0]
print("\nColumns:", list(sample.keys()))
print("\nExample record:")
for k, v in sample.items():
    print(f"  {k}: {str(v)[:200]}")


README.md:   0%|          | 0.00/622 [00:00<?, ?B/s]

data/train-00000-of-00001-d16079d82b6163(…):   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/test-00000-of-00001-055b08f98c91e8f(…):   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11400 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2136 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 11400
    })
    test: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2136
    })
})

Columns: ['input', 'output', 'instruction']

Example record:
  input: NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns about his health on Monday, saying his weight loss was caused by a hormone imbalance that is relatively sim
  output: founded_by: Apple Inc, Steve Jobs; chief_executive_officer: Apple Inc, Steve Jobs
  instruction: Given phrases that describe the relationship between two words/phrases as options, extract the word/phrase pair and the corresponding lexical relationship between them from the input text. The output 


## 4. Build train / validation / test splits and prompt format

Each example is turned into an instruction-tuning prompt:

```
### Instruction:
<task instruction>

### Input:
<sentence + entity pair>

### Response:
<relation label>
```

Training labels mask out the prompt tokens (loss computed only on the `### Response:` continuation).

In [ ]:
DEFAULT_INSTRUCTION = (
    "Given the financial sentence and the two marked entities, identify the "
    "relation that holds between them. Respond with the relation label only."
)

def get_field(ex, *names, default=""):
    for n in names:
        if n in ex and ex[n]:
            return ex[n]
    return default

def build_prompt(ex):
    instruction = get_field(ex, "instruction", default=DEFAULT_INSTRUCTION)
    inp         = get_field(ex, "input", "sentence", "text", default="")
    target      = get_field(ex, "output", "relation", "label", default="")
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
    return {"prompt": prompt, "target": str(target).strip()}

full_ds = raw[split_name].map(build_prompt, remove_columns=raw[split_name].column_names)
full_ds = full_ds.filter(lambda ex: len(ex["target"]) > 0 and len(ex["prompt"]) > 0)


if set(raw.keys()) >= {"train", "validation", "test"}:
    train_ds = raw["train"].map(build_prompt, remove_columns=raw["train"].column_names)
    val_ds   = raw["validation"].map(build_prompt, remove_columns=raw["validation"].column_names)
    test_ds  = raw["test"].map(build_prompt, remove_columns=raw["test"].column_names)
else:
    split1  = full_ds.train_test_split(test_size=0.2, seed=SEED)
    split2  = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    train_ds, val_ds, test_ds = split1["train"], split2["train"], split2["test"]

if MAX_TRAIN_SAMPLES:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
if MAX_EVAL_SAMPLES:
    test_ds = test_ds.shuffle(seed=SEED).select(range(min(MAX_EVAL_SAMPLES, len(test_ds))))

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print("\nSample prompt:\n", train_ds[0]["prompt"], "\n-->", train_ds[0]["target"])


Map:   0%|          | 0/11400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11400 [00:00<?, ? examples/s]

train=9120  val=1140  test=500

Sample prompt:
 ### Instruction:
Given phrases that describe the relationship between two words/phrases as options, extract the word/phrase pair and the corresponding lexical relationship between them from the input text. The output format should be "relation1: word1, word2; relation2: word3, word4". Options: product/material produced, manufacturer, distributed by, industry, position held, original broadcaster, owned by, founded by, distribution format, headquarters location, stock exchange, currency, parent organization, chief executive officer, director/manager, owner of, operator, member of, employer, chairperson, platform, subsidiary, legal form, publisher, developer, brand, business division, location of formation, creator.

### Input:
(MENAFN - Arab News) Lulu Hypermarket described as the lifestyle shopping destination of choice for many and the largest retail operator in the region has geared up for the Eid Al-Fitr celebration with major promotion

## 5. Tokenizer and tokenization (prompt tokens masked with -100)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_example(ex):
    prompt_ids = tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(ex["target"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + target_ids)[:MAX_LEN]
    labels    = ([-100] * len(prompt_ids) + target_ids)[:MAX_LEN]
    attn_mask = [1] * len(input_ids)

    
    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

train_tok.set_format(type="torch")
val_tok.set_format(type="torch")


_lengths = [len(x) for x in train_tok["input_ids"]]
print(f"Tokenized length stats -- min: {min(_lengths)}, mean: {sum(_lengths)/len(_lengths):.1f}, "
      f"p95: {sorted(_lengths)[int(0.95*len(_lengths))]}, max: {max(_lengths)} (MAX_LEN={MAX_LEN})")


## 6. Load Qwen2.5-3B in 4-bit NF4 (QLoRA base model)

The quantized base model is loaded **once** and reused across every rank in the sweep — only the LoRA adapter (attached with `get_peft_model`) changes between runs, then is stripped with `model.unload()` before the next rank so the base weights are untouched.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # fp16 on T4/P100, bf16 only where hardware-supported
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit NF4 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP,   # single GPU -- avoid needless multi-GPU model sharding
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING
)
base_model.config.use_cache = False
print("Base model loaded.")


Loading base model in 4-bit NF4 ...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded.


## 7. Helper functions: LoRA config, param/memory accounting, F1 evaluation

In [ ]:
def make_lora_config(r):
    return LoraConfig(
        r=r,
        lora_alpha=r * LORA_ALPHA_MULT,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

def count_trainable_params(model):
    trainable, total = 0, 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return trainable, total

def adapter_memory_mb(model):
    """Size of just the LoRA adapter weights, at their stored dtype."""
    total_bytes = 0
    for name, p in model.named_parameters():
        if "lora_" in name and p.requires_grad:
            total_bytes += p.numel() * p.element_size()
    return total_bytes / (1024 ** 2)

def adapter_disk_mb(adapter_dir):
    total = 0
    for f in os.listdir(adapter_dir):
        fp = os.path.join(adapter_dir, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return total / (1024 ** 2)

def extract_relation(generated_text):
    # take the first non-empty line after generation as the predicted relation
    for line in generated_text.strip().split("\n"):
        line = line.strip()
        if line:
            return line
    return ""

@torch.no_grad()
def evaluate_f1(model, tokenizer, eval_dataset, max_new_tokens=16, gen_batch_size=8):
    model.eval()
    preds, golds = [], []
    for i in range(0, len(eval_dataset), gen_batch_size):
        batch = eval_dataset[i:i + gen_batch_size]
        prompts = batch["prompt"]
        golds.extend(batch["target"])

        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                         truncation=True, max_length=MAX_LEN).to(model.device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        gen_only = out[:, enc["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        preds.extend(extract_relation(t) for t in decoded)

    # Normalize casing/whitespace for a fair label match, then micro-F1 over relation labels
    norm_preds = [p.strip().lower() for p in preds]
    norm_golds = [g.strip().lower() for g in golds]
    labels = sorted(set(norm_golds) | set(norm_preds))
    label2id = {l: i for i, l in enumerate(labels)}
    y_true = [label2id[g] for g in norm_golds]
    y_pred = [label2id[p] for p in norm_preds]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    return {"precision": precision, "recall": recall, "f1": f1,
            "n_eval": len(norm_golds)}


## 8. Rank sweep: train + measure params / memory / F1

In [ ]:
r = LORA_RANKS  # your chosen LoRA rank

print(f"\n{'='*60}\nLoRA rank r={r}\n{'='*60}")
torch.cuda.empty_cache(); gc.collect()
try:
    torch.cuda.reset_peak_memory_stats()
except AttributeError:
    print("warning: torch.cuda.reset_peak_memory_stats unavailable; peak_gpu_mem_mb will read as 0")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100
)

lora_config = make_lora_config(r)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable_params, total_params = count_trainable_params(model)
run_dir = os.path.join(OUTPUT_ROOT, f"r{r}")
os.makedirs(run_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=run_dir,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    logging_steps=25,
    save_strategy="no",
    report_to=[],
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60

try:
    peak_gpu_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
except AttributeError:
    peak_gpu_mem_mb = 0.0

adapter_dir = os.path.join(run_dir, "adapter")
model.save_pretrained(adapter_dir)

mem_tensor_mb = adapter_memory_mb(model)
mem_disk_mb   = adapter_disk_mb(adapter_dir)

metrics = evaluate_f1(model, tokenizer, test_ds)

result = {
    "rank": r,
    "alpha": r * LORA_ALPHA_MULT,
    "trainable_params": trainable_params,
    "trainable_pct": 100 * trainable_params / total_params,
    "adapter_mem_tensor_mb": mem_tensor_mb,
    "adapter_mem_disk_mb": mem_disk_mb,
    "peak_gpu_mem_mb": peak_gpu_mem_mb,
    "train_time_min": train_time_min,
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "n_eval": metrics["n_eval"],
}
print(result)


## 9. Results table

In [ ]:
results_df = pd.DataFrame([result])
results_df["f1"] = results_df["f1"].round(4)
results_df["precision"] = results_df["precision"].round(4)
results_df["recall"] = results_df["recall"].round(4)
results_df["adapter_mem_tensor_mb"] = results_df["adapter_mem_tensor_mb"].round(2)
results_df["adapter_mem_disk_mb"] = results_df["adapter_mem_disk_mb"].round(2)
results_df["peak_gpu_mem_mb"] = results_df["peak_gpu_mem_mb"].round(1)
results_df